# GA, PSO, DE, and hybrid feature selection on two biomedical tables

This notebook is self-contained. Attach the Kaggle datasets `kmader/skin-cancer-mnist-ham10000` and `alitaqishah/blood-cell-anomaly-detection-2025`. It evaluates **full features, GA, PSO, DE, GA + PSO, and GA + DE** on HAM10000 and the blood-cell anomaly table. No Fashion-MNIST is used.

The outer test records never guide feature selection. HAM10000 uses lesion-group splits; blood cells use stratified rows because patient IDs are unavailable. The default is three shared 750/300 splits, two feature penalties, eight candidates, and six generations. Search CPU time and selected feature names are saved for every run.


In [ ]:
from pathlib import Path
import os, shutil
root = Path.cwd()
inputs = Path('/kaggle/input')
required = {
    'HAM10000_metadata.csv': root / 'data' / 'ham',
    'hmnist_8_8_RGB.csv': root / 'data' / 'ham',
    'blood_cell_anomaly_detection.csv': root / 'data' / 'blood',
}
for filename, destination in required.items():
    target = destination / filename
    if not target.exists():
        matches = list(inputs.rglob(filename)) if inputs.exists() else []
        if len(matches) != 1:
            raise FileNotFoundError(f'Expected one {filename}; found {len(matches)}. Attach both datasets or place local files under data/.')
        destination.mkdir(parents=True, exist_ok=True)
        shutil.copy2(matches[0], target)
print('Input files ready:', [str(d / f) for f, d in required.items()])


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, log_loss, roc_auc_score
DATA = Path.cwd() / 'data'
def load_dataset(name: str):
    if name == "HAM10000":
        pixels = pd.read_csv(DATA / "ham" / "hmnist_8_8_RGB.csv")
        meta = pd.read_csv(DATA / "ham" / "HAM10000_metadata.csv")
        mapping = {"akiec": 0, "bcc": 1, "bkl": 2, "df": 3,
                   "nv": 4, "vasc": 5, "mel": 6}
        if len(pixels) != len(meta) or not np.array_equal(
            pixels["label"].to_numpy(), meta["dx"].map(mapping).to_numpy()
        ):
            raise ValueError("HAM pixel/metadata row alignment failed")
        x = pixels.drop(columns="label").to_numpy(dtype=np.float32) / 255.0
        y = (meta["dx"] == "mel").astype(int).to_numpy()
        groups = meta["lesion_id"].astype(str).to_numpy()
        return x, y, groups, list(pixels.columns[:-1])
    if name == "Blood cell":
        df = pd.read_csv(DATA / "blood" / "blood_cell_anomaly_detection.csv")
        # Restrict to measured cell morphology, colour, and routine blood counts.
        # Exclude cell_type, disease_category, and three model/labeller scores.
        features = ["cell_diameter_um", "nucleus_area_pct", "chromatin_density",
                    "cytoplasm_ratio", "circularity", "eccentricity",
                    "granularity_score", "lobularity_score", "membrane_smoothness",
                    "cell_area_px", "perimeter_px", "mean_r", "mean_g", "mean_b",
                    "stain_intensity", "wbc_count_per_ul",
                    "rbc_count_millions_per_ul", "hemoglobin_g_dl",
                    "hematocrit_pct", "platelet_count_per_ul", "mcv_fl", "mchc_g_dl"]
        if df["cell_id"].duplicated().any() or df[features].isna().any().any():
            raise ValueError("Unexpected duplicate ID or missing blood feature")
        return df[features].to_numpy(dtype=np.float32), df["anomaly_label"].to_numpy(dtype=int), None, features
    raise ValueError(name)

def balanced_subsample(indices, labels, n, seed):
    if len(indices) <= n:
        return indices
    chosen, _ = train_test_split(indices, train_size=n, stratify=labels[indices], random_state=seed)
    return np.sort(chosen)

def split_data(x, y, groups, seed, train_n, test_n):
    idx = np.arange(len(y))
    if groups is None:
        tr, te = train_test_split(idx, test_size=0.28, stratify=y, random_state=seed)
    else:
        splitter = GroupShuffleSplit(n_splits=1, test_size=0.28, random_state=seed)
        tr, te = next(splitter.split(x, y, groups))
    tr = balanced_subsample(tr, y, train_n, seed)
    te = balanced_subsample(te, y, test_n, seed + 100)
    if groups is not None and set(groups[tr]) & set(groups[te]):
        raise AssertionError("Lesion leakage")
    if len(np.unique(y[tr])) < 2 or len(np.unique(y[te])) < 2:
        raise ValueError("Split has only one class")
    scaler = StandardScaler().fit(x[tr])
    return scaler.transform(x[tr]), scaler.transform(x[te]), y[tr], y[te], tr, te

def binary_metrics(y, prediction, probability):
    tn, fp, fn, tp = confusion_matrix(y, prediction, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if tp + fn else np.nan
    specificity = tn / (tn + fp) if tn + fp else np.nan
    precision = tp / (tp + fp) if tp + fp else np.nan
    npv = tn / (tn + fn) if tn + fn else np.nan
    f1 = 2 * precision * sensitivity / (precision + sensitivity) if precision + sensitivity else np.nan
    acc = (tp + tn) / len(y)
    prevalence = (tp + fn) / len(y)
    predicted = (tp + fp) / len(y)
    chance = prevalence * predicted + (1 - prevalence) * (1 - predicted)
    return dict(tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp),
                accuracy=acc, error=1-acc, sensitivity=sensitivity,
                specificity=specificity, precision=precision, recall=sensitivity,
                f_measure=f1, auc_roc=roc_auc_score(y, probability),
                kappa=(acc-chance)/(1-chance),
                informedness=sensitivity+specificity-1,
                markedness=precision+npv-1,
                log_loss=log_loss(y, probability, labels=[0, 1]))

## Search protocol

Each feature mask is evaluated by two inner folds on the training partition. The fitness is inner ROC AUC minus `penalty × fraction of features selected`. GA uses crossover and mutation; PSO uses personal and global best masks; DE mutates and crosses real-valued vectors before thresholding. Both hybrids run three GA generations followed by three generations of the named second optimizer. Every search makes 56 attempted fitness evaluations, with repeated masks cached.


In [ ]:
"""Leakage-aware wrapper feature selection on the two AML datasets.

GA, binary PSO, real-coded DE, GA->PSO, and GA->DE share the same fitness
function and attempted evaluation budget. The outer test split is untouched
until a feature mask has been selected using inner training folds.
"""
from __future__ import annotations

import argparse
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.special import expit
from scipy.stats import ttest_rel, wilcoxon
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits


ROOT = Path.cwd()
METHODS = ("Full", "GA", "PSO", "DE", "GA + PSO", "GA + DE")


def repair(mask: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    mask = np.asarray(mask, dtype=bool).copy()
    if mask.sum() < 2:
        mask[rng.choice(mask.size, size=2, replace=False)] = True
    return mask


def initial_population(rng, size, features):
    return np.vstack([repair(rng.random(features) < rng.uniform(.15, .55), rng)
                      for _ in range(size)])


class Fitness:
    def __init__(self, x, y, groups, seed, penalty):
        self.x, self.y, self.penalty = x, y, penalty
        if groups is None:
            cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=seed)
            self.folds = list(cv.split(x, y))
        else:
            cv = StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=seed)
            self.folds = list(cv.split(x, y, groups))
            assert all(set(groups[tr]).isdisjoint(groups[va]) for tr, va in self.folds)
        self.cache = {}
        self.attempts = 0

    def __call__(self, mask):
        self.attempts += 1
        key = np.asarray(mask, dtype=bool).tobytes()
        if key in self.cache:
            return self.cache[key]
        if np.count_nonzero(mask) < 2:
            return -np.inf
        scores = []
        for tr, va in self.folds:
            if len(np.unique(self.y[tr])) < 2 or len(np.unique(self.y[va])) < 2:
                return -np.inf
            scaler = StandardScaler().fit(self.x[tr][:, mask])
            xt = scaler.transform(self.x[tr][:, mask])
            xv = scaler.transform(self.x[va][:, mask])
            clf = LogisticRegression(class_weight="balanced", max_iter=1000,
                                     random_state=42)
            clf.fit(xt, self.y[tr])
            from sklearn.metrics import roc_auc_score
            scores.append(roc_auc_score(self.y[va], clf.predict_proba(xv)[:, 1]))
        result = (float(np.mean(scores) - self.penalty * np.mean(mask)),
                  float(np.mean(scores)))
        self.cache[key] = result
        return result


def optimize(method, fitness, seed, population_size=8, generations=6):
    """Equal attempted evaluations: population_size * (generations + 1)."""
    rng = np.random.default_rng(seed)
    d = fitness.x.shape[1]
    population = initial_population(rng, population_size, d)
    scores = np.array([fitness(mask)[0] for mask in population])
    best_idx = int(np.argmax(scores))
    best, best_score = population[best_idx].copy(), float(scores[best_idx])
    trajectory = [dict(generation=0, best_fitness=best_score,
                       best_features=int(best.sum()))]
    velocities = rng.normal(0, .3, size=(population_size, d))
    personal = population.copy()
    personal_scores = scores.copy()
    real_population = np.where(population, .75, .25).astype(float)

    for generation in range(1, generations + 1):
        phase = method
        if method == "GA + PSO":
            phase = "GA" if generation <= generations // 2 else "PSO"
        elif method == "GA + DE":
            phase = "GA" if generation <= generations // 2 else "DE"
        candidates = []
        if phase == "GA":
            candidates.append(best.copy())
            while len(candidates) < population_size:
                i, j = rng.integers(population_size, size=2)
                a = population[i] if scores[i] >= scores[j] else population[j]
                i, j = rng.integers(population_size, size=2)
                b = population[i] if scores[i] >= scores[j] else population[j]
                child = np.where(rng.random(d) < .5, a, b)
                child = np.logical_xor(child, rng.random(d) < max(1 / d, .02))
                candidates.append(repair(child, rng))
        elif phase == "PSO":
            velocities = (.55 * velocities + 1.3 * rng.random((population_size, d)) *
                          (personal.astype(float) - population.astype(float)) +
                          1.3 * rng.random((population_size, d)) *
                          (best.astype(float) - population.astype(float)))
            velocities = np.clip(velocities, -4, 4)
            candidates = [repair(rng.random(d) < expit(v), rng)
                          for v in velocities]
        elif phase == "DE":
            for i in range(population_size):
                a, b, c = rng.choice([j for j in range(population_size) if j != i],
                                     size=3, replace=False)
                donor = np.clip(real_population[a] + .6 *
                                (real_population[b] - real_population[c]), 0, 1)
                crossing = rng.random(d) < .7
                crossing[rng.integers(d)] = True
                trial_real = np.where(crossing, donor, real_population[i])
                candidates.append(repair(trial_real >= .5, rng))
        else:
            raise ValueError(method)
        candidate_masks = np.vstack(candidates)
        candidate_scores = np.array([fitness(mask)[0] for mask in candidate_masks])
        if phase == "DE":
            keep = candidate_scores > scores
            population[keep] = candidate_masks[keep]
            scores[keep] = candidate_scores[keep]
            real_population[keep] = np.where(population[keep], .75, .25)
        else:
            population, scores = candidate_masks, candidate_scores
            real_population = np.where(population, .75, .25).astype(float)
        better = scores > personal_scores
        personal[better], personal_scores[better] = population[better], scores[better]
        idx = int(np.argmax(scores))
        if scores[idx] > best_score:
            best, best_score = population[idx].copy(), float(scores[idx])
        trajectory.append(dict(generation=generation, best_fitness=best_score,
                               best_features=int(best.sum())))
    assert fitness.attempts == population_size * (generations + 1)
    return best, trajectory


def evaluate(xtr, ytr, xte, yte, mask, seed):
    scaler = StandardScaler().fit(xtr[:, mask])
    clf = LogisticRegression(class_weight="balanced", max_iter=1000,
                             random_state=seed)
    clf.fit(scaler.transform(xtr[:, mask]), ytr)
    probability = clf.predict_proba(scaler.transform(xte[:, mask]))[:, 1]
    prediction = (probability >= .5).astype(int)
    return binary_metrics(yte, prediction, probability), probability, prediction


def run(args):
    out = Path(args.output)
    out.mkdir(parents=True, exist_ok=True)
    rows, traces, selections, predictions = [], [], [], []
    penalties = [float(v) for v in args.penalties.split(",")]
    if not penalties or any(v < 0 for v in penalties):
        raise ValueError("Penalties must be nonnegative")
    for dataset in ("HAM10000", "Blood cell"):
        x, y, groups, feature_names = load_dataset(dataset)
        for seed in range(args.seeds):
            split_seed = 100 + seed
            _, _, ytr, yte, tr, te = split_data(x, y, groups, split_seed,
                                                args.train_n, args.test_n)
            xtr, xte = x[tr], x[te]
            inner_groups = None if groups is None else groups[tr]
            for method in METHODS:
                settings = [None] if method == "Full" else penalties
                for setting, penalty in enumerate(settings):
                    start_cpu = time.process_time()
                    with threadpool_limits(limits=1):
                        if method == "Full":
                            mask = np.ones(x.shape[1], dtype=bool)
                            trajectory = []
                            fitness = None
                        else:
                            fitness = Fitness(xtr, ytr, inner_groups,
                                              split_seed, penalty)
                            mask, trajectory = optimize(method, fitness,
                                                       10000 * (seed + 1) +
                                                       100 * setting + METHODS.index(method),
                                                       args.population, args.generations)
                        metrics, prob, pred = evaluate(xtr, ytr, xte, yte,
                                                       mask, split_seed)
                    cpu = time.process_time() - start_cpu
                    inner_auc = (np.nan if fitness is None else
                                 fitness.cache[mask.tobytes()][1])
                    row = dict(dataset=dataset, seed=seed, method=method,
                               setting=setting, sparsity_penalty=penalty,
                               n_train=len(tr), n_test=len(te),
                               n_features=int(mask.sum()), feature_fraction=float(mask.mean()),
                               evaluations=0 if fitness is None else fitness.attempts,
                               unique_masks=0 if fitness is None else len(fitness.cache),
                               inner_auc=inner_auc,
                               cpu_seconds=cpu, **metrics)
                    rows.append(row)
                    selections.append(dict(dataset=dataset, seed=seed,
                                           method=method, setting=setting,
                                           features=json.dumps([feature_names[i]
                                                                for i in np.flatnonzero(mask)])))
                    for point in trajectory:
                        traces.append(dict(dataset=dataset, seed=seed,
                                           method=method, setting=setting, **point))
                    predictions.extend(dict(dataset=dataset, seed=seed,
                                            method=method, setting=setting,
                                            test_index=int(te[i]), y=int(yte[i]),
                                            probability=float(prob[i]), pred=int(pred[i]))
                                       for i in range(len(te)))
                    print(f"{dataset} seed={seed} {method} penalty={penalty}: "
                          f"AUC={metrics['auc_roc']:.3f}, features={mask.sum()}, CPU={cpu:.1f}s",
                          flush=True)
    df = pd.DataFrame(rows)
    df.to_csv(out / "runs.csv", index=False)
    pd.DataFrame(traces).to_csv(out / "trajectories.csv", index=False)
    pd.DataFrame(selections).to_csv(out / "selected_features.csv", index=False)
    pd.DataFrame(predictions).to_csv(out / "predictions.csv", index=False)
    primary = df[(df.method == "Full") | (df.setting == 0)]
    primary.groupby(["dataset", "method"], sort=False).agg(
        auc_roc_mean=("auc_roc", "mean"), auc_roc_sd=("auc_roc", "std"),
        accuracy_mean=("accuracy", "mean"), sensitivity_mean=("sensitivity", "mean"),
        specificity_mean=("specificity", "mean"), precision_mean=("precision", "mean"),
        f_measure_mean=("f_measure", "mean"), log_loss_mean=("log_loss", "mean"),
        selected_features_mean=("n_features", "mean"),
        cpu_seconds_mean=("cpu_seconds", "mean")).reset_index().to_csv(
            out / "primary_summary.csv", index=False)
    tests = []
    for dataset in ("HAM10000", "Blood cell"):
        base = primary[(primary.dataset == dataset) & (primary.method == "Full")].sort_values("seed")
        for method in METHODS[1:]:
            other = primary[(primary.dataset == dataset) & (primary.method == method)].sort_values("seed")
            d = other.auc_roc.to_numpy() - base.auc_roc.to_numpy()
            tests.append(dict(dataset=dataset, method=method,
                              mean_auc_delta=float(d.mean()),
                              paired_t_p=float(ttest_rel(other.auc_roc, base.auc_roc).pvalue),
                              wilcoxon_p=float(wilcoxon(d).pvalue)))
    pd.DataFrame(tests).to_csv(out / "exploratory_tests.csv", index=False)
    (out / "protocol.json").write_text(json.dumps(dict(
        datasets=["HAM10000", "Blood cell"], methods=METHODS,
        seeds=args.seeds, train_n=args.train_n, test_n=args.test_n,
        penalties=penalties, population=args.population,
        generations=args.generations, inner_folds=2,
        objective="mean inner ROC AUC - penalty * feature fraction",
        threshold=.5, group_split="lesion_id for HAM10000; rows for blood cell",
        note="Exploratory comparison; three overlapping splits are not independent."
    ), indent=2), encoding="utf-8")




In [ ]:
"""Figures for the measured feature-selection extension."""
from __future__ import annotations

import argparse
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve

METHODS = ["Full", "GA", "PSO", "DE", "GA + PSO", "GA + DE"]
COLORS = dict(zip(METHODS, ["#153B50", "#4E8A9E", "#D17A42", "#72935B",
                            "#8B6AA8", "#BA5C5C"]))
DATASETS = ["HAM10000", "Blood cell"]


def save(fig, path):
    fig.savefig(path, dpi=190, bbox_inches="tight", facecolor="white")
    plt.close(fig)


def paired_bars(summary, column, ylabel, path, log=False):
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 5), sharey=False)
    for ax, dataset in zip(axes, DATASETS):
        part = summary[summary.dataset == dataset].set_index("method").reindex(METHODS)
        vals = part[column].to_numpy()
        ax.bar(np.arange(len(METHODS)), vals, color=[COLORS[m] for m in METHODS])
        if log:
            ax.set_yscale("log")
            ax.set_ylim(max(min(vals) * .75, .005), max(vals) * 1.55)
        ax.set_xticks(range(len(METHODS)), METHODS, rotation=35, ha="right")
        ax.set_title(dataset, fontsize=15)
        ax.set_ylabel(ylabel)
        ax.grid(axis="y", alpha=.2)
        ax.set_axisbelow(True)
        for i, v in enumerate(vals):
            label = (f"{v:.3f}" if column in ("auc_roc_mean", "accuracy_mean", "log_loss_mean") else
                     f"{v:.2f}" if log else f"{v:.1f}")
            ax.text(i, v * (1.08 if log else 1.01), label, ha="center", fontsize=8)
        if column == "auc_roc_mean":
            ax.set_ylim(0, max(vals) + .1)
        elif column == "selected_features_mean":
            ax.set_ylim(0, max(vals) * 1.17)
    fig.tight_layout()
    save(fig, path)


def make(out):
    out = Path(out)
    figures = out / "figures"
    figures.mkdir(exist_ok=True)
    runs = pd.read_csv(out / "runs.csv")
    summary = pd.read_csv(out / "primary_summary.csv")
    traces = pd.read_csv(out / "trajectories.csv")
    selected = pd.read_csv(out / "selected_features.csv")
    predictions = pd.read_csv(out / "predictions.csv")
    paired_bars(summary, "auc_roc_mean", "Mean held-out ROC AUC", figures / "auc_comparison.png")
    paired_bars(summary, "accuracy_mean", "Mean held-out accuracy", figures / "accuracy_comparison.png")
    paired_bars(summary, "log_loss_mean", "Mean held-out log loss (lower is better)",
                figures / "loss_comparison.png")
    paired_bars(summary, "selected_features_mean", "Selected features (mean)",
                figures / "feature_count.png")
    paired_bars(summary, "cpu_seconds_mean", "Process CPU seconds (log scale)",
                figures / "cpu_comparison.png", log=True)

    fig, axes = plt.subplots(1, 2, figsize=(12.5, 5))
    for ax, dataset in zip(axes, DATASETS):
        part = summary[summary.dataset == dataset].set_index("method")
        for m in METHODS:
            row = part.loc[m]
            ax.scatter(row.selected_features_mean, row.auc_roc_mean,
                       color=COLORS[m], s=105, label=m)
            ax.annotate(m, (row.selected_features_mean, row.auc_roc_mean),
                        xytext=(5, 4), textcoords="offset points", fontsize=8)
        ax.set_title(dataset)
        ax.set_xlabel("Mean feature count")
        ax.set_ylabel("Mean held-out ROC AUC")
        ax.grid(alpha=.2)
    fig.tight_layout()
    save(fig, figures / "auc_vs_features.png")

    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))
    for ax, dataset in zip(axes, DATASETS):
        part = traces[(traces.dataset == dataset) & (traces.setting == 0)]
        for m in METHODS[1:]:
            mean = part[part.method == m].groupby("generation").best_fitness.mean()
            ax.plot(mean.index, mean.values, marker="o", markersize=3,
                    color=COLORS[m], label=m)
        ax.set_title(dataset)
        ax.set_xlabel("Generation")
        ax.set_ylabel("Best inner fitness (mean of 3 searches)")
        ax.grid(alpha=.2)
    axes[1].legend(loc="lower right", fontsize=8)
    fig.tight_layout()
    save(fig, figures / "search_convergence.png")

    for dataset in DATASETS:
        fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
        part = predictions[(predictions.dataset == dataset) &
                           (predictions.seed == 0) & (predictions.setting == 0)]
        for m in METHODS:
            d = part[part.method == m]
            fpr, tpr, _ = roc_curve(d.y, d.probability)
            mean_auc = summary[(summary.dataset == dataset) &
                               (summary.method == m)].auc_roc_mean.iloc[0]
            axes[0].plot(fpr, tpr, color=COLORS[m],
                         label=f"{m} (mean {mean_auc:.3f})")
        axes[0].plot([0, 1], [0, 1], ls="--", color="gray", alpha=.5)
        axes[0].set(xlabel="False positive rate", ylabel="Sensitivity",
                    title="ROC curves: first held-out split")
        axes[0].legend(fontsize=8)
        best = summary[summary.dataset == dataset].sort_values("auc_roc_mean").iloc[-1].method
        for j, m in enumerate(("Full", best)):
            d = part[part.method == m]
            tn = int(((d.y == 0) & (d.pred == 0)).sum())
            fp = int(((d.y == 0) & (d.pred == 1)).sum())
            fn = int(((d.y == 1) & (d.pred == 0)).sum())
            tp = int(((d.y == 1) & (d.pred == 1)).sum())
            axes[1].text(.08, .77 - j * .43,
                         f"{m}\nTN {tn}   FP {fp}\nFN {fn}   TP {tp}",
                         transform=axes[1].transAxes, fontsize=15,
                         va="top", color=COLORS[m])
        axes[1].set(title="Confusion counts at 0.5 threshold")
        axes[1].axis("off")
        fig.tight_layout()
        save(fig, figures / f"{dataset.replace(' ', '_')}_roc_confusion.png")

    for dataset in DATASETS:
        part = selected[(selected.dataset == dataset) &
                        (selected.setting == 0) & (selected.method != "Full")]
        counts = {}
        for encoded in part.features:
            for feature in json.loads(encoded):
                counts[feature] = counts.get(feature, 0) + 1
        top = sorted(counts.items(), key=lambda kv: kv[1], reverse=True)[:18]
        fig, ax = plt.subplots(figsize=(9.4, 5.2))
        ax.barh([p[0] for p in top][::-1], [p[1] for p in top][::-1],
                color="#4E8A9E")
        ax.set_xlabel("Selections among 15 primary searches (5 methods × 3 seeds)")
        ax.set_title(f"{dataset}: frequently selected features")
        ax.grid(axis="x", alpha=.2)
        fig.tight_layout()
        save(fig, figures / f"{dataset.replace(' ', '_')}_feature_frequency.png")

    sensitivity = runs[runs.method != "Full"].groupby(
        ["dataset", "method", "sparsity_penalty"], as_index=False).agg(
            auc=("auc_roc", "mean"), features=("n_features", "mean"))
    sensitivity.to_csv(out / "parameter_summary.csv", index=False)
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8))
    for ax, dataset in zip(axes, DATASETS):
        part = sensitivity[sensitivity.dataset == dataset]
        for method in METHODS[1:]:
            p = part[part.method == method].sort_values("sparsity_penalty")
            ax.plot(p.sparsity_penalty, p.auc, marker="o", color=COLORS[method],
                    label=method)
        ax.set(title=dataset, xlabel="Feature-count penalty",
               ylabel="Mean held-out ROC AUC")
        ax.grid(alpha=.2)
    axes[1].legend(fontsize=8)
    fig.tight_layout()
    save(fig, figures / "parameter_sensitivity.png")




In [ ]:
from argparse import Namespace
settings = Namespace(
    seeds=int(os.getenv('AML_FS_SEEDS', '3')),
    train_n=int(os.getenv('AML_FS_TRAIN_N', '750')),
    test_n=int(os.getenv('AML_FS_TEST_N', '300')),
    penalties=os.getenv('AML_FS_PENALTIES', '0.02,0.05'),
    population=int(os.getenv('AML_FS_POPULATION', '8')),
    generations=int(os.getenv('AML_FS_GENERATIONS', '6')),
    output=str(root / 'aml-feature-selection' / 'results'),
)
run(settings)
make(settings.output)


In [ ]:
from IPython.display import display, Image
result_dir = Path(settings.output)
display(pd.read_csv(result_dir / 'primary_summary.csv').round(3))
display(pd.read_csv(result_dir / 'parameter_summary.csv').round(3))
display(pd.read_csv(result_dir / 'exploratory_tests.csv').round(3))
for name in ['auc_comparison.png', 'accuracy_comparison.png', 'loss_comparison.png',
             'feature_count.png', 'cpu_comparison.png',
             'auc_vs_features.png', 'search_convergence.png',
             'parameter_sensitivity.png', 'HAM10000_roc_confusion.png',
             'Blood_cell_roc_confusion.png', 'HAM10000_feature_frequency.png',
             'Blood_cell_feature_frequency.png']:
    display(Image(filename=str(result_dir / 'figures' / name)))


## Interpretation

Compare AUC, sensitivity, precision, feature count, and CPU time together. A smaller mask is useful only if the held-out performance and study goal justify the added search cost. Three overlapping splits and small inner folds make the exploratory p-values weak evidence. Do not choose the better penalty using the held-out test results. FROC AUC is unavailable because these classification tables do not have scored candidate lesion locations and false positives per image. The 8 × 8 lesion pixels and missing blood-cell patient IDs prevent clinical generalization claims.
